# White blood cell classifier – live demo
Upload or pick a cell image and see the prediction, the class probabilities and a Grad-CAM heatmap.
Uses the **final model**: the SmallCNN selected by validation on the leak-free group split (`experiments_groupsplit/smallcnn_scratch_s42`). Example images are **clean test images**: their source photo was never in training.

**A CPU runtime is enough** (Runtime → Change runtime type → CPU or T4). Run the cells top to bottom; allow ~5 minutes the first time. On presentation day, start this notebook **15 minutes early** and keep the tab open.

## 1. Code and packages

In [ ]:
import os
REPO = "https://github.com/Yasmin-maker1/Automated-Classification-of-White-Blood-Cell-Subtypes-Using-Convolutional-Neural-Networks.git"
if os.path.exists("/content/repo/.git"):
    %cd /content/repo
    !git pull -q
else:
    !git clone -q {REPO} /content/repo
    %cd /content/repo
!pip -q install gradio

## 2. Trained model and split file from Google Drive
Both were saved by `colab_quickstart.ipynb` (checkpoints are not stored in GitHub). Any other run works too, e.g. `densenet169_ft_s42`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wbc_project"
CKPT = f"{DRIVE}/experiments_groupsplit/smallcnn_scratch_s42/best.pt"
SPLIT = f"{DRIVE}/group_split.json"
for p in (CKPT, SPLIT):
    print(("found    " if os.path.exists(p) else "MISSING  ") + p)

## 3. Example images (first time only)
Picks 2 clean test images per class, each from a different photo. Needs the data set: if `data/` is missing, upload `kaggle.json` when asked. Skip this cell if `demo/examples/` already has images.

In [ ]:
if not os.path.exists("data/dataset2-master"):
    from google.colab import files
    files.upload()   # kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d paultimothymooney/blood-cells -p data --unzip
!python demo/prepare_demo.py --data-root data --checkpoint {CKPT} --split-file {SPLIT}
from IPython.display import Image, display
display(Image("results/figures/demo_showcase.png"))

## 4. Launch the demo
A link appears below (`Running on public URL: ...`). Open it, or use the inline window. Stop the cell (square button) when you are done.

In [ ]:
!python demo/app.py --checkpoint {CKPT} --share

## 5. Keep the showcase figure and examples (optional)
Downloads a zip with the static showcase figure (backup slide) and the example images.

In [ ]:
!zip -qr demo_files.zip results/figures/demo_showcase.png demo/examples
from google.colab import files
files.download("demo_files.zip")